# Паренклитика высшего порядка: тест на Parity-$k$ датасетах

## Модели и подходы

### `MyModel` — паренклитическая сеть (генеративная)

Каждое условное распределение $p(x_i \mid x_I)$ аппроксимируется гауссовой линейной регрессией.
Это _генеративная_ модель: мы умеем оценивать $\log p(x)$.

**Два нативных способа классификации:**

| Подход | Идея | Замечание |
|---|---|---|
| **Contrastive Parenclitic (LLR)** | Обучаем $M_0$ на классе 0 и $M_1$ на классе 1; $\text{score}(x)=\log p_1(x) - \log p_0(x)$ | Полностью нативный, без внешнего классификатора |
| **Parenclitic Features** | Обучаем $M$ только на классе 0 (норма); признаки $= \text{get\_features}(x)$ → внешний классификатор | Аномальность = несоответствие "нормальной" модели |

### `MyModelSynolitic` — синолитическая сеть (дискриминативная)

Каждый классификатор $p(y \mid x_T)$ обучается напрямую на обоих классах.
Предсказание собирается через формулу Мёбиуса:
$$\log p(y|x) \approx_k \sum_{t=0}^{k} c^{(k)}(t,d) \sum_{T:|T|=t} \log p(y|x_T) + A_{d,k}\log p(y)$$

| Подход | Идея |
|---|---|
| **Synolitic Native** | Прямой `predict_proba()` — Мёбиусова формула без внешнего классификатора |
| **Synolitic Features** | `get_feature_matrix_full_aggregated` → внешний классификатор |

### Принцип сравнения

- **Нативные**: LLR vs Synolitic Native — без внешних классификаторов
- **Features**: Parenclitic Features vs Synolitic Features — **одинаковый** внешний классификатор


In [1]:
import numpy as np
import sys
sys.path.insert(0, '/home/ernest/Desktop/диплом')

from models import MyModel
from models_synolitic import MyModelSynolitic

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.base import clone as sklearn_clone

import plotly.graph_objects as go
from plotly.subplots import make_subplots

print('Imports OK')

Imports OK


## 1. Генераторы Parity-$k$


In [2]:
def generate_parity_k(n, d, k, seed=42):
    """Parity-k: d // k непересекающихся блоков, метка = majority vote."""
    assert d % k == 0
    num_blocks = d // k
    rng = np.random.default_rng(seed)
    X = rng.uniform(-1, 1, size=(n, d))
    S = np.array([
        np.sign(np.prod(X[:, i*k:(i+1)*k], axis=1))
        for i in range(num_blocks)
    ]).T                             # (n, num_blocks)
    Y = np.sign(S.sum(axis=1))
    mask = Y != 0
    return X[mask], ((Y[mask] + 1) / 2).astype(int)


def generate_mixed_parity(components, n, weights=None, seed=42):
    """components: list of (k, d_component) or (k, d_component, weight)."""
    parsed = [(c[0], c[1], c[2] if len(c) > 2 else 1.0) for c in components]
    if weights is not None:
        parsed = [(k_, d_, w) for (k_, d_, _), w in zip(parsed, weights)]
    rng = np.random.default_rng(seed)
    d_total = sum(d_ for k_, d_, _ in parsed)
    X = rng.uniform(-1, 1, size=(n, d_total))
    score = np.zeros(n)
    col = 0
    for k_, d_, w_ in parsed:
        for i in range(d_ // k_):
            score += w_ * np.sign(np.prod(X[:, col:col+k_], axis=1))
            col += k_
    Y = np.sign(score)
    mask = Y != 0
    return X[mask], ((Y[mask] + 1) / 2).astype(int)


for k_, d_, n_ in [(2, 2, 500), (3, 3, 1000), (4, 4, 3000)]:
    X_, y_ = generate_parity_k(n_, d_, k_)
    print(f'Parity-{k_}  n={len(y_):<5} balance={np.mean(y_):.3f}')

Parity-2  n=500   balance=0.510
Parity-3  n=1000  balance=0.470
Parity-4  n=3000  balance=0.487


## 2. Функции оценки

### 2.1 Parenclitic: Contrastive (LLR)

Обучаем отдельную `MyModel` на каждом классе. Скор: $\log p_1(x) - \log p_0(x)$.

### 2.2 Parenclitic: Features (fit on class 0)

Обучаем `MyModel` **только на классе 0** (норма). Затем применяем к всем объектам,
получая признаки «насколько x не похож на норму». Признаки → внешний классификатор.

### 2.3 Synolitic: Native

Прямой `predict_proba()` из синолитики (Мёбиусова формула).

### 2.4 Synolitic: Features

`get_feature_matrix_full_aggregated` → внешний классификатор (тот же, что в 2.2).


In [3]:
# ────────────────────────────────────────────────────────────────────
# Вспомогательные утилиты
# ────────────────────────────────────────────────────────────────────

def _make_ext_clf(clf_spec):
    """clf_spec: sklearn estimator или Pipeline — возвращает обёрнутый в Pipeline+Scaler."""
    if clf_spec is None:
        clf_spec = LogisticRegression(max_iter=2000)
    if isinstance(clf_spec, Pipeline):
        return clf_spec
    return Pipeline([('scaler', StandardScaler()), ('clf', clf_spec)])


def _get_parenclitic_features(model, X, extraction):
    """extraction: 'aggregated' | 'full' | 'raw'"""
    if extraction == 'full':
        F, _ = model.get_feature_matrix_full(X)
    elif extraction == 'raw':
        F = model.get_feature_matrix(X)
        if not isinstance(F, np.ndarray):
            F = F[0]
    else:  # aggregated (default)
        F, _ = model.get_feature_matrix_full_aggregated(X)
    return F


def _get_synolitic_features(model, X):
    F, _ = model.get_feature_matrix_full_aggregated(X)
    return F


# ────────────────────────────────────────────────────────────────────
# Оценка: одна CV-fold, возвращает dict {'llr', 'par_feat', 'syn_nat', 'syn_feat'}
# ────────────────────────────────────────────────────────────────────

def _evaluate_one_split(
    X_tr, y_tr, X_te, y_te,
    k_test,                         # порядок модели
    ext_clf_spec,                   # внешний классификатор (или None → LogReg)
    par_extraction,                 # тип агрегации признаков паренклитики
    syn_clf_class, syn_clf_params,  # внутренний классификатор синолитики
    run_llr=True, run_par_feat=True,
    run_syn_nat=True, run_syn_feat=True,
):
    d = X_tr.shape[1]
    aucs = {}

    # ── Contrastive Parenclitic (LLR) ───────────────────────────────
    if run_llr:
        try:
            m0 = MyModel(d=d, k=k_test)
            m1 = MyModel(d=d, k=k_test)
            m0.fit_parallel(X_tr[y_tr == 0], n_jobs=-1)
            m1.fit_parallel(X_tr[y_tr == 1], n_jobs=-1)
            score = m1.log_prob(X_te) - m0.log_prob(X_te)
            aucs['llr'] = roc_auc_score(y_te, score)
        except Exception as e:
            aucs['llr'] = 0.5
            print(f'  [LLR] k={k_test}: {e}')

    # ── Parenclitic Features (fit on class 0) ───────────────────────
    if run_par_feat:
        try:
            m0 = MyModel(d=d, k=k_test)
            m0.fit_parallel(X_tr[y_tr == 0], n_jobs=-1)
            F_tr = _get_parenclitic_features(m0, X_tr, par_extraction)
            F_te = _get_parenclitic_features(m0, X_te, par_extraction)
            clf = sklearn_clone(_make_ext_clf(ext_clf_spec))
            clf.fit(F_tr, y_tr)
            aucs['par_feat'] = roc_auc_score(y_te, clf.predict_proba(F_te)[:, 1])
        except Exception as e:
            aucs['par_feat'] = 0.5
            print(f'  [ParFeat] k={k_test}: {e}')

    # ── Synolitic Native ────────────────────────────────────────────
    if run_syn_nat:
        try:
            ms = MyModelSynolitic(
                d=d, k=k_test,
                classifier_class=syn_clf_class,
                clf_class_params=syn_clf_params,
            )
            ms.fit_parallel(X_tr, y_tr, n_jobs=-1)
            proba = ms.predict_proba(X_te)
            class1_idx = list(ms.classes_).index(1)
            aucs['syn_nat'] = roc_auc_score(y_te, proba[:, class1_idx])
        except Exception as e:
            aucs['syn_nat'] = 0.5
            print(f'  [SynNat] k={k_test}: {e}')

    # ── Synolitic Features ──────────────────────────────────────────
    if run_syn_feat:
        try:
            ms = MyModelSynolitic(
                d=d, k=k_test,
                classifier_class=syn_clf_class,
                clf_class_params=syn_clf_params,
            )
            ms.fit_parallel(X_tr, y_tr, n_jobs=-1)
            F_tr = _get_synolitic_features(ms, X_tr)
            F_te = _get_synolitic_features(ms, X_te)
            clf = sklearn_clone(_make_ext_clf(ext_clf_spec))
            clf.fit(F_tr, y_tr)
            aucs['syn_feat'] = roc_auc_score(y_te, clf.predict_proba(F_te)[:, 1])
        except Exception as e:
            aucs['syn_feat'] = 0.5
            print(f'  [SynFeat] k={k_test}: {e}')

    return aucs


def evaluate_all(
    X, y,
    k_values,
    n_repeats=5,
    test_size=0.3,
    ext_clf_spec=None,              # внешний clf для Features-подходов (оба)
    par_extraction='aggregated',    # агрегация паренклитических признаков
    syn_clf_class=None,             # внутренний clf синолитики (None = LogReg)
    syn_clf_params=None,
    run_llr=True, run_par_feat=True,
    run_syn_nat=True, run_syn_feat=True,
):
    """
    Оценка AUC для 4 подходов на сетке k_values.

    Возвращает:
        { подход: { k: {'auc_mean': ..., 'auc_std': ...} } }

    Подходы: 'llr', 'par_feat', 'syn_nat', 'syn_feat'
    """
    from sklearn.linear_model import LogisticRegression
    if syn_clf_class is None:
        syn_clf_class = LogisticRegression
    if syn_clf_params is None:
        syn_clf_params = {'max_iter': 1000}

    raw = {ap: {k: [] for k in k_values}
           for ap in ('llr', 'par_feat', 'syn_nat', 'syn_feat')}

    for seed in range(n_repeats):
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=test_size, random_state=42 + seed, stratify=y
        )
        for k in k_values:
            fold = _evaluate_one_split(
                X_tr, y_tr, X_te, y_te,
                k_test=k,
                ext_clf_spec=ext_clf_spec,
                par_extraction=par_extraction,
                syn_clf_class=syn_clf_class,
                syn_clf_params=syn_clf_params,
                run_llr=run_llr, run_par_feat=run_par_feat,
                run_syn_nat=run_syn_nat, run_syn_feat=run_syn_feat,
            )
            for ap, v in fold.items():
                raw[ap][k].append(v)

    return {
        ap: {k: {'auc_mean': np.mean(vals), 'auc_std': np.std(vals)}
             for k, vals in kd.items()}
        for ap, kd in raw.items()
    }


print('Функции определены.')

Функции определены.


## 3. Вспомогательные функции визуализации


In [4]:
def hex_to_rgba(hex_color, alpha=0.5):
    h = hex_color.lstrip('#')
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f'rgba({r},{g},{b},{alpha})'


# Стили линий для 4 подходов
APPROACH_STYLE = {
    'llr':      dict(color='#2980b9', dash='solid',  symbol='circle',
                     label='Contrastive Parenclitic (LLR)'),
    'par_feat': dict(color='#27ae60', dash='dash',   symbol='square',
                     label='Parenclitic Features (fit class 0)'),
    'syn_nat':  dict(color='#e74c3c', dash='dot',    symbol='diamond',
                     label='Synolitic Native (Möbius)'),
    'syn_feat': dict(color='#e67e22', dash='dashdot',symbol='triangle-up',
                     label='Synolitic Features'),
}


def plot_auc_sweep(results_dict, title, k_values,
                   approaches=None, dataset_names=None):
    """
    results_dict: { dataset_name: { approach: { k: {auc_mean, auc_std} } } }
    """
    if dataset_names is None:
        dataset_names = list(results_dict.keys())
    if approaches is None:
        approaches = list(APPROACH_STYLE.keys())

    fig = make_subplots(
        rows=1, cols=len(dataset_names),
        subplot_titles=dataset_names,
        shared_yaxes=True,
    )

    for col_idx, ds in enumerate(dataset_names, start=1):
        res = results_dict[ds]
        x_labels = [f'тест. k = {k}' for k in k_values]

        for ap in approaches:
            if ap not in res:
                continue
            data = res[ap]
            st = APPROACH_STYLE[ap]
            means = [data[k]['auc_mean'] for k in k_values]
            stds  = [data[k]['auc_std']  for k in k_values]

            fig.add_trace(
                go.Scatter(
                    x=x_labels, y=means,
                    error_y=dict(type='data', array=stds, visible=True),
                    mode='lines+markers',
                    name=st['label'],
                    line=dict(color=st['color'], width=2.5, dash=st['dash']),
                    marker=dict(size=11, symbol=st['symbol'], color=st['color']),
                    showlegend=(col_idx == 1),
                ),
                row=1, col=col_idx,
            )

        fig.add_hline(y=0.5, line_dash='dot', line_color='gray',
                      line_width=1.5, row=1, col=col_idx)

    fig.update_layout(
        title=dict(text=title, x=0.5, font_size=14),
        yaxis=dict(title='AUC (ROC)', range=[0.3, 1.08]),
        plot_bgcolor='white', height=480,
        legend=dict(x=1.01, y=0.98, font_size=11),
        font=dict(size=12),
    )
    for col_idx in range(1, len(dataset_names) + 1):
        fig.update_yaxes(
            showgrid=True, gridcolor='rgba(200,200,200,0.4)',
            range=[0.3, 1.08], row=1, col=col_idx,
        )
    fig.update_xaxes(showgrid=False)
    fig.show()


print('Вспомогательные функции визуализации готовы.')

Вспомогательные функции визуализации готовы.


## 4. Sweep тестового $k$: Parity-2, 3, 4

**Конфигурация**: $d = k_\text{датасета}$ — один блок.

**Внутренний clf синолитики**: LogisticRegression (baseline).

**Внешний clf** для Features-подходов: тот же LogisticRegression.


In [5]:
# Конфигурации Parity-k
PARITY_CONFIGS = [
    (2, 2,  500,  'Parity-2'),
    (3, 3, 1000,  'Parity-3'),
    (4, 4, 3000,  'Parity-4'),
]

sweep_results = {}

for k_ds, d, n, name in PARITY_CONFIGS:
    print(f'\n── {name} (n={n}, d={d}) ──')
    X_, y_ = generate_parity_k(n=n, d=d, k=k_ds, seed=42)
    res = evaluate_all(
        X_, y_,
        k_values=list(range(1, d + 1)),
        n_repeats=5,
        ext_clf_spec=None,               # LogReg (default)
        par_extraction='aggregated',
        syn_clf_class=None,              # LogReg (default)
    )
    sweep_results[name] = res

    for ap, data in res.items():
        row = '  '.join(
            f'k={k}: {data[k]["auc_mean"]:.3f}'
            for k in range(1, d + 1)
        )
        print(f'  [{APPROACH_STYLE[ap]["label"][:28]}] {row}')

print('\nГотово!')


── Parity-2 (n=500, d=2) ──
  [Contrastive Parenclitic (LLR] k=1: 0.996  k=2: 0.996
  [Parenclitic Features (fit cl] k=1: 0.994  k=2: 0.994
  [Synolitic Native (Möbius)] k=1: 0.462  k=2: 0.465
  [Synolitic Features] k=1: 0.464  k=2: 0.645

── Parity-3 (n=1000, d=3) ──
  [Contrastive Parenclitic (LLR] k=1: 0.494  k=2: 0.491  k=3: 0.491
  [Parenclitic Features (fit cl] k=1: 0.549  k=2: 0.548  k=3: 0.548
  [Synolitic Native (Möbius)] k=1: 0.497  k=2: 0.498  k=3: 0.498
  [Synolitic Features] k=1: 0.498  k=2: 0.508  k=3: 0.511

── Parity-4 (n=3000, d=4) ──
  [Contrastive Parenclitic (LLR] k=1: 0.492  k=2: 0.496  k=3: 0.499  k=4: 0.499
  [Parenclitic Features (fit cl] k=1: 0.845  k=2: 0.892  k=3: 0.892  k=4: 0.892
  [Synolitic Native (Möbius)] k=1: 0.487  k=2: 0.486  k=3: 0.486  k=4: 0.486
  [Synolitic Features] k=1: 0.486  k=2: 0.509  k=3: 0.514  k=4: 0.513

Готово!


In [6]:
# Отображаем все 4 подхода
results_for_plot = {
    name: sweep_results[name]
    for _, _, _, name in PARITY_CONFIGS
}

for k_ds, d, n, name in PARITY_CONFIGS:
    plot_auc_sweep(
        {name: sweep_results[name]},
        title=f'<b>{name}: AUC vs тестовое k — все 4 подхода</b>',
        k_values=list(range(1, d + 1)),
        dataset_names=[name],
    )

## 5. Влияние $d$ при $k_\text{датасета}=3$

При росте $d$ увеличивается число информативных блоков (сигнал суммируется),
но каждый блок несёт меньший вес в majority vote.


In [7]:
D_VALUES = [3, 9, 15, 21]
dsweep_results = {}

for d_val in D_VALUES:
    name = f'Parity-3 (d={d_val})'
    print(f'{name}...', end=' ', flush=True)
    X_, y_ = generate_parity_k(n=1000, d=d_val, k=3, seed=42)
    res = evaluate_all(
        X_, y_, k_values=[1, 2, 3], n_repeats=5,
        run_llr=True, run_par_feat=True, run_syn_nat=True, run_syn_feat=True
    )
    dsweep_results[name] = res
    row = '  '.join(
        f'k={k}: LLR={res["llr"][k]["auc_mean"]:.3f} SynNat={res["syn_nat"][k]["auc_mean"]:.3f}'
        for k in [1, 2, 3]
    )
    print(row)

print('Готово!')

Parity-3 (d=3)... k=1: LLR=0.494 SynNat=0.497  k=2: LLR=0.491 SynNat=0.498  k=3: LLR=0.491 SynNat=0.498
Parity-3 (d=9)... k=1: LLR=0.490 SynNat=0.473  k=2: LLR=0.501 SynNat=0.476  k=3: LLR=0.506 SynNat=0.477
Parity-3 (d=15)... k=1: LLR=0.517 SynNat=0.503  k=2: LLR=0.522 SynNat=0.502  k=3: LLR=0.526 SynNat=0.496
Parity-3 (d=21)... k=1: LLR=0.533 SynNat=0.539  k=2: LLR=0.538 SynNat=0.539  k=3: LLR=0.540 SynNat=0.542
Готово!


In [8]:
# Два субплота: LLR и Synolitic Native для разных d
fig_d = make_subplots(rows=1, cols=2,
                      subplot_titles=['Contrastive Parenclitic (LLR)',
                                      'Synolitic Native'],
                      shared_yaxes=True)

d_colors = {3: '#e74c3c', 9: '#e67e22', 15: '#27ae60', 21: '#2980b9'}

for d_val in D_VALUES:
    name = f'Parity-3 (d={d_val})'
    color = d_colors[d_val]
    x_lab = ['тест. k=1', 'тест. k=2', 'тест. k=3']

    for col_idx, ap in enumerate(['llr', 'syn_nat'], start=1):
        data = dsweep_results[name][ap]
        means = [data[k]['auc_mean'] for k in [1, 2, 3]]
        stds  = [data[k]['auc_std']  for k in [1, 2, 3]]
        fig_d.add_trace(
            go.Scatter(
                x=x_lab, y=means,
                error_y=dict(type='data', array=stds, visible=True),
                mode='lines+markers',
                name=f'd={d_val}',
                line=dict(color=color, width=2.5),
                marker=dict(size=10),
                showlegend=(col_idx == 1),
            ),
            row=1, col=col_idx,
        )

for col_idx in [1, 2]:
    fig_d.add_hline(y=0.5, line_dash='dot', line_color='gray',
                    line_width=1.5, row=1, col=col_idx)

fig_d.update_layout(
    title=dict(text='<b>Parity-3 при разном d: нативные подходы</b>', x=0.5, font_size=14),
    yaxis=dict(title='AUC (ROC)', range=[0.4, 0.6]),
    plot_bgcolor='white', height=450,
    legend=dict(title='Число признаков d', x=1.01, y=0.98),
    font=dict(size=12),
)
fig_d.update_xaxes(showgrid=False)
fig_d.update_yaxes(showgrid=True, gridcolor='rgba(200,200,200,0.4)')
fig_d.show()

## 6. Mixed Parity: сравнение паренклитики и синолитики на нескольких ML-моделях

Сравниваем все 4 подхода на каждом Mixed-датасете при 3 внешних / внутренних классификаторах:

| # | Классификатор | Обозначение |
|---|---|---|
| 1 | `LogisticRegression` | **LogReg** — линейный |
| 2 | `SVC(kernel='rbf', C=10)` | **RBF-SVM** — нелинейный |
| 3 | `GradientBoostingClassifier` | **GBM** — ансамбль деревьев |

> **Нативные подходы** (LLR и Synolitic Native) запускаются один раз — они не зависят от внешнего классификатора.


In [9]:
from sklearn.ensemble import GradientBoostingClassifier

MIXED_CONFIGS = [
    {'label': 'Mixed P2+P3', 'components': [(2, 2), (3, 3)],
     'n': 2000, 'k_values': [1, 2, 3], 'k1': 2, 'k2': 3, 'color': '#9b59b6'},
    {'label': 'Mixed P3+P4', 'components': [(3, 3), (4, 4)],
     'n': 3000, 'k_values': [1, 2, 3, 4], 'k1': 3, 'k2': 4, 'color': '#e67e22'},
]

# Конфигурации ML-моделей
ML_CONFIGS = [
    {
        'name': 'LogReg',
        'ext_clf': None,   # LogReg (default)
        'syn_clf_class': None,
        'syn_clf_params': None,
    },
    {
        'name': 'RBF-SVM',
        'ext_clf': SVC(kernel='rbf', probability=True, C=10.0),
        'syn_clf_class': SVC,
        'syn_clf_params': {'kernel': 'rbf', 'probability': True, 'C': 10.0},
    },
    {
        'name': 'GBM',
        'ext_clf': GradientBoostingClassifier(n_estimators=100),
        'syn_clf_class': GradientBoostingClassifier,
        'syn_clf_params': {'n_estimators': 100},
    },
]

mixed_results = {}   # label → { ml_name → eval_all_result }

for cfg in MIXED_CONFIGS:
    mixed_results[cfg['label']] = {}
    X_, y_ = generate_mixed_parity(cfg['components'], n=cfg['n'], seed=42)

    for ml in ML_CONFIGS:
        print(f"{cfg['label']} / {ml['name']}...", end=' ', flush=True)
        res = evaluate_all(
            X_, y_,
            k_values=cfg['k_values'],
            n_repeats=5,
            ext_clf_spec=ml['ext_clf'],
            syn_clf_class=ml['syn_clf_class'],
            syn_clf_params=ml['syn_clf_params'],
            run_llr=(ml['name'] == 'LogReg'),    # нативные только 1 раз
            run_syn_nat=(ml['name'] == 'LogReg'),
            run_par_feat=True,
            run_syn_feat=True,
        )
        mixed_results[cfg['label']][ml['name']] = res
        # При первом прогоне сохраняем нативные
        if ml['name'] == 'LogReg':
            mixed_results[cfg['label']]['_native'] = {
                'llr': res['llr'], 'syn_nat': res['syn_nat']
            }
        print('OK')

print('\nВсе Mixed-эксперименты завершены!')


Mixed P2+P3 / LogReg... OK
Mixed P2+P3 / RBF-SVM... OK
Mixed P2+P3 / GBM... 

/home/ernest/anaconda3/envs/recsys-2025/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/ernest/anaconda3/envs/recsys-2025/lib/python3.9/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/ernest/anaconda3/envs/recsys-2025/lib/python3.9/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/ernest/anaconda3/envs/recsys-2025/lib/python3.9/site-packages/numpy/core/_methods.py:163: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/home/ernest/anaconda3/envs/recsys-2025/lib/python3.9/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


OK
Mixed P3+P4 / LogReg... 

/home/ernest/anaconda3/envs/recsys-2025/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/ernest/anaconda3/envs/recsys-2025/lib/python3.9/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/ernest/anaconda3/envs/recsys-2025/lib/python3.9/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/ernest/anaconda3/envs/recsys-2025/lib/python3.9/site-packages/numpy/core/_methods.py:163: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/home/ernest/anaconda3/envs/recsys-2025/lib/python3.9/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


OK
Mixed P3+P4 / RBF-SVM... OK
Mixed P3+P4 / GBM... 

/home/ernest/anaconda3/envs/recsys-2025/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/ernest/anaconda3/envs/recsys-2025/lib/python3.9/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/ernest/anaconda3/envs/recsys-2025/lib/python3.9/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/ernest/anaconda3/envs/recsys-2025/lib/python3.9/site-packages/numpy/core/_methods.py:163: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/home/ernest/anaconda3/envs/recsys-2025/lib/python3.9/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


OK

Все Mixed-эксперименты завершены!


/home/ernest/anaconda3/envs/recsys-2025/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/ernest/anaconda3/envs/recsys-2025/lib/python3.9/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/ernest/anaconda3/envs/recsys-2025/lib/python3.9/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/ernest/anaconda3/envs/recsys-2025/lib/python3.9/site-packages/numpy/core/_methods.py:163: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/home/ernest/anaconda3/envs/recsys-2025/lib/python3.9/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [10]:
# ── Столбчатые диаграммы: для каждого Mixed-датасета — один subplot per k,
#    сгруппированные столбцы по подходам × ML-модели
# ──────────────────────────────────────────────────────────────────────────

AP_LABELS = {
    'llr':      'Contrastive LLR',
    'syn_nat':  'Synolitic Native',
    'par_feat': 'Parenclitic Feat.',
    'syn_feat': 'Synolitic Feat.',
}

# Цвета: по ML-модели; прозрачность: по подходу
ML_COLOR  = {'LogReg': '#3498db', 'RBF-SVM': '#e74c3c', 'GBM': '#27ae60'}
AP_ALPHA  = {'llr': 1.0, 'syn_nat': 0.65, 'par_feat': 0.90, 'syn_feat': 0.50}
AP_HATCH  = {'llr': '', 'syn_nat': '/', 'par_feat': '', 'syn_feat': '/'}

for cfg in MIXED_CONFIGS:
    ds = cfg['label']
    k_vals = cfg['k_values']
    n_k = len(k_vals)

    fig = make_subplots(
        rows=1, cols=n_k,
        subplot_titles=[f'тест. k = {k}' for k in k_vals],
        shared_yaxes=True,
    )

    # Все комбо: (ml_name, ap) исключая нативные с non-LogReg
    bars = []
    native = mixed_results[ds]['_native']
    for ap in ['llr', 'syn_nat']:
        bars.append(('Native', ap, native[ap]))
    for ml in ML_CONFIGS:
        for ap in ['par_feat', 'syn_feat']:
            bars.append((ml['name'], ap, mixed_results[ds][ml['name']][ap]))

    # Строим по одному trace на комбо
    shown = set()
    for bar_ml, ap, data in bars:
        color_key = bar_ml if bar_ml != 'Native' else (
            '#2980b9' if ap == 'llr' else '#c0392b'
        )
        color = ML_COLOR.get(bar_ml, color_key)
        alpha = AP_ALPHA[ap]
        rgba_color = hex_to_rgba(color, alpha) if alpha < 1.0 else color

        trace_name = f"{AP_LABELS[ap]} [{bar_ml}]"

        for col_idx, k in enumerate(k_vals, start=1):
            v   = data[k]['auc_mean']
            std = data[k]['auc_std']
            fig.add_trace(
                go.Bar(
                    name=trace_name,
                    x=[trace_name],
                    y=[v],
                    error_y=dict(type='data', array=[std], visible=True),
                    marker_color=rgba_color,
                    marker_line_color='white', marker_line_width=1,
                    text=[f'{v:.3f}'],
                    textposition='outside',
                    showlegend=(col_idx == 1 and trace_name not in shown),
                ),
                row=1, col=col_idx,
            )
            shown.add(trace_name)

    for col_idx in range(1, n_k + 1):
        fig.add_hline(y=0.5, line_dash='dot', line_color='gray',
                      line_width=1.5, row=1, col=col_idx)

    fig.update_layout(
        title=dict(
            text=f'<b>{ds}: AUC по подходам и ML-моделям</b><br>'
                 '<sup>Нативные (LLR, SynNat) не зависят от ML | '
                 'Features-подходы — LogReg / RBF-SVM / GBM</sup>',
            x=0.5, font_size=13,
        ),
        yaxis=dict(title='AUC (ROC)', range=[0.3, 1.25]),
        plot_bgcolor='white',
        height=520,
        legend=dict(x=1.01, y=1.0, font_size=10),
        font=dict(size=11),
        barmode='group',
        bargap=0.15, bargroupgap=0.05,
    )
    fig.update_xaxes(showgrid=False, showticklabels=False)
    for col_idx in range(1, n_k + 1):
        fig.update_yaxes(
            showgrid=True, gridcolor='rgba(200,200,200,0.4)',
            range=[0.3, 1.25], row=1, col=col_idx,
        )
    fig.show()


In [11]:
# ── Сводная таблица в Markdown ─────────────────────────────────────────────
from IPython.display import Markdown, display

lines = []

for cfg in MIXED_CONFIGS:
    ds = cfg['label']
    k_vals = cfg['k_values']
    native = mixed_results[ds]['_native']

    lines.append(f'### {ds}\n')
    # Header
    k_header = ' | '.join(f'тест. k={k}' for k in k_vals)
    lines.append(f'| Подход | Классификатор | {k_header} |')
    sep = '|---|---|' + '|'.join(['---'] * len(k_vals)) + '|'
    lines.append(sep)

    # Нативные (независимо от ML)
    for ap, ap_label in [('llr', 'Contrastive LLR'), ('syn_nat', 'Synolitic Native')]:
        vals = ' | '.join(
            f'**{native[ap][k]["auc_mean"]:.3f}** ±{native[ap][k]["auc_std"]:.3f}'
            for k in k_vals
        )
        lines.append(f'| {ap_label} | — (нативный) | {vals} |')

    # Features-подходы по ML
    for ap, ap_label in [('par_feat', 'Parenclitic Feat.'), ('syn_feat', 'Synolitic Feat.')]:
        for ml in ML_CONFIGS:
            data = mixed_results[ds][ml['name']][ap]
            vals = ' | '.join(
                f'{data[k]["auc_mean"]:.3f} ±{data[k]["auc_std"]:.3f}'
                for k in k_vals
            )
            lines.append(f'| {ap_label} | {ml["name"]} | {vals} |')

    lines.append('')

display(Markdown('\n'.join(lines)))


### Mixed P2+P3

| Подход | Классификатор | тест. k=1 | тест. k=2 | тест. k=3 |
|---|---|---|---|---|
| Contrastive LLR | — (нативный) | **0.990** ±0.005 | **0.994** ±0.002 | **0.995** ±0.002 |
| Synolitic Native | — (нативный) | **0.476** ±0.029 | **0.477** ±0.028 | **0.477** ±0.029 |
| Parenclitic Feat. | LogReg | 0.997 ±0.002 | 0.997 ±0.001 | 0.997 ±0.002 |
| Parenclitic Feat. | RBF-SVM | 0.989 ±0.003 | 0.991 ±0.002 | 0.992 ±0.002 |
| Parenclitic Feat. | GBM | 0.984 ±0.004 | 0.983 ±0.004 | 0.983 ±0.005 |
| Synolitic Feat. | LogReg | 0.477 ±0.028 | 0.784 ±0.066 | 0.830 ±0.063 |
| Synolitic Feat. | RBF-SVM | 0.489 ±0.034 | 0.998 ±0.002 | 1.000 ±0.000 |
| Synolitic Feat. | GBM | 0.462 ±0.024 | 0.997 ±0.003 | 0.997 ±0.003 |

### Mixed P3+P4

| Подход | Классификатор | тест. k=1 | тест. k=2 | тест. k=3 | тест. k=4 |
|---|---|---|---|---|---|
| Contrastive LLR | — (нативный) | **0.515** ±0.017 | **0.519** ±0.017 | **0.521** ±0.017 | **0.522** ±0.017 |
| Synolitic Native | — (нативный) | **0.511** ±0.010 | **0.513** ±0.011 | **0.513** ±0.011 | **0.513** ±0.011 |
| Parenclitic Feat. | LogReg | 0.531 ±0.030 | 0.543 ±0.031 | 0.550 ±0.032 | 0.556 ±0.032 |
| Parenclitic Feat. | RBF-SVM | 0.488 ±0.007 | 0.490 ±0.010 | 0.498 ±0.013 | 0.501 ±0.014 |
| Parenclitic Feat. | GBM | 0.503 ±0.030 | 0.515 ±0.039 | 0.517 ±0.026 | 0.511 ±0.037 |
| Synolitic Feat. | LogReg | 0.511 ±0.011 | 0.592 ±0.038 | 0.609 ±0.043 | 0.622 ±0.043 |
| Synolitic Feat. | RBF-SVM | 0.501 ±0.013 | 0.538 ±0.028 | 0.982 ±0.007 | 0.995 ±0.003 |
| Synolitic Feat. | GBM | 0.476 ±0.011 | 0.489 ±0.002 | 0.538 ±0.079 | 0.579 ±0.101 |


In [12]:
# ── Heatmap: строки = (датасет × подход × ML), столбцы = тестовое k ──────

max_k = max(len(cfg['k_values']) for cfg in MIXED_CONFIGS)
all_k = sorted(set(k for cfg in MIXED_CONFIGS for k in cfg['k_values']))

heat_rows = []

for cfg in MIXED_CONFIGS:
    ds = cfg['label']
    k_vals = cfg['k_values']
    native = mixed_results[ds]['_native']

    for ap, ap_label in [('llr', 'LLR'), ('syn_nat', 'SynNat')]:
        row_z = [native[ap][k]['auc_mean'] if k in k_vals else None for k in all_k]
        row_t = [f"{native[ap][k]['auc_mean']:.3f}" if k in k_vals else '—' for k in all_k]
        heat_rows.append({'label': f'{ds} / {ap_label} (native)', 'z': row_z, 't': row_t})

    for ap, ap_short in [('par_feat', 'ParFeat'), ('syn_feat', 'SynFeat')]:
        for ml in ML_CONFIGS:
            data = mixed_results[ds][ml['name']][ap]
            row_z = [data[k]['auc_mean'] if k in k_vals else None for k in all_k]
            row_t = [f"{data[k]['auc_mean']:.3f}" if k in k_vals else '—' for k in all_k]
            heat_rows.append({
                'label': f'{ds} / {ap_short} [{ml["name"]}]',
                'z': row_z, 't': row_t,
            })

fig_heat = go.Figure(go.Heatmap(
    z=[r['z'] for r in heat_rows],
    x=[f'тест. k={k}' for k in all_k],
    y=[r['label'] for r in heat_rows],
    text=[r['t'] for r in heat_rows],
    texttemplate='%{text}', textfont=dict(size=11),
    colorscale=[[0, '#d9534f'], [0.35, '#f7f7f7'], [1, '#2ecc71']],
    zmin=0.45, zmax=1.0,
    colorbar=dict(title='AUC', tickformat='.2f'),
))

fig_heat.update_layout(
    title=dict(
        text='<b>Mixed Parity Heatmap: AUC(тест. k) × подход × ML-модель</b><br>'
             '<sup>Красный = слепой (AUC≈0.5) | Зелёный = информативный</sup>',
        x=0.5, font_size=13,
    ),
    xaxis=dict(side='top'),
    height=680,
    font=dict(size=11),
)
fig_heat.show()


## 7. Прямое сравнение: нативные подходы vs Features с одинаковым классификатором

Запускаем на Parity-3 ($d=3$, $n=1000$) три конфигурации:

| Config | Внешний clf | Внутренний clf синолитики |
|---|---|---|
| **Baseline** | LogReg | LogReg |
| **SVM** | RBF-SVM | RBF-SVM |
| **Нативные** | — | LogReg (Мёбиус) |


In [13]:
X_p3, y_p3 = generate_parity_k(n=1000, d=3, k=3, seed=42)

comparison = {}

# Config 1: LogReg внешний + LogReg внутренний
print('Config: LogReg...', end=' ', flush=True)
comparison['LogReg'] = evaluate_all(
    X_p3, y_p3, k_values=[1, 2, 3], n_repeats=5,
    ext_clf_spec=None,
    syn_clf_class=None,
)
print('OK')

# Config 2: RBF-SVM внешний + RBF-SVM внутренний
print('Config: RBF-SVM...', end=' ', flush=True)
svm_clf = SVC(kernel='rbf', probability=True, C=10.0)
comparison['RBF-SVM'] = evaluate_all(
    X_p3, y_p3, k_values=[1, 2, 3], n_repeats=5,
    ext_clf_spec=svm_clf,
    syn_clf_class=SVC,
    syn_clf_params={'kernel': 'rbf', 'probability': True, 'C': 10.0},
)
print('OK')

# Сводная таблица
print('\n── Сводная таблица AUC при тест. k=3 ──')
header = f'{"Подход":<38} {"LogReg":>10} {"RBF-SVM":>10}'
print(header)
print('-' * 60)
for ap, style in APPROACH_STYLE.items():
    v_lr  = comparison['LogReg'][ap][3]['auc_mean']
    v_svm = comparison['RBF-SVM'][ap][3]['auc_mean']
    print(f'{style["label"]:<38} {v_lr:>10.3f} {v_svm:>10.3f}')

Config: LogReg... OK
Config: RBF-SVM... OK

── Сводная таблица AUC при тест. k=3 ──
Подход                                     LogReg    RBF-SVM
------------------------------------------------------------
Contrastive Parenclitic (LLR)               0.491      0.491
Parenclitic Features (fit class 0)          0.548      0.506
Synolitic Native (Möbius)                   0.498      0.991
Synolitic Features                          0.511      0.988


In [14]:
fig_cmp = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Внешний clf: LogReg', 'Внешний clf: RBF-SVM'],
    shared_yaxes=True,
)

k_vals = [1, 2, 3]
x_labels = [f'тест. k = {k}' for k in k_vals]

for col_idx, config_name in enumerate(['LogReg', 'RBF-SVM'], start=1):
    res = comparison[config_name]
    for ap, style in APPROACH_STYLE.items():
        data = res[ap]
        means = [data[k]['auc_mean'] for k in k_vals]
        stds  = [data[k]['auc_std']  for k in k_vals]
        fig_cmp.add_trace(
            go.Scatter(
                x=x_labels, y=means,
                error_y=dict(type='data', array=stds, visible=True),
                mode='lines+markers',
                name=style['label'],
                line=dict(color=style['color'], width=2.5, dash=style['dash']),
                marker=dict(size=11, symbol=style['symbol'], color=style['color']),
                showlegend=(col_idx == 1),
            ),
            row=1, col=col_idx,
        )
    fig_cmp.add_hline(y=0.5, line_dash='dot', line_color='gray',
                      line_width=1.5, row=1, col=col_idx)

fig_cmp.update_layout(
    title=dict(
        text='<b>Parity-3: сравнение 4 подходов при LogReg и RBF-SVM</b>',
        x=0.5, font_size=14,
    ),
    yaxis=dict(title='AUC (ROC)', range=[0.3, 1.08]),
    plot_bgcolor='white', height=480,
    legend=dict(x=1.01, y=0.98, font_size=11),
    font=dict(size=12),
)
fig_cmp.update_xaxes(showgrid=False)
fig_cmp.update_yaxes(showgrid=True, gridcolor='rgba(200,200,200,0.4)')
fig_cmp.show()

## 8. Сводный Heatmap


In [15]:
APPROACHES_FOR_HEATMAP = ['llr', 'syn_nat', 'par_feat', 'syn_feat']

# Строки: dataset × approach
heatmap_rows = []

all_ds = [(k_ds, d, n, name) for k_ds, d, n, name in PARITY_CONFIGS]

for k_ds, d, n, name in all_ds:
    res = sweep_results[name]
    for ap in APPROACHES_FOR_HEATMAP:
        if ap in res:
            row_vals = []
            for k in range(1, d + 1):
                if k in res[ap]:
                    row_vals.append(res[ap][k]['auc_mean'])
                else:
                    row_vals.append(None)
            # Pad to 4
            while len(row_vals) < 4:
                row_vals.append(None)
            heatmap_rows.append({
                'label': f"{name} / {APPROACH_STYLE[ap]['label'][:22]}",
                'vals': row_vals,
            })

fig_heat = go.Figure(go.Heatmap(
    z=[r['vals'] for r in heatmap_rows],
    x=[f'тест. k={k}' for k in range(1, 5)],
    y=[r['label'] for r in heatmap_rows],
    text=[[f'{v:.3f}' if v is not None else '—' for v in r['vals']]
          for r in heatmap_rows],
    texttemplate='%{text}', textfont=dict(size=11),
    colorscale=[[0, '#d9534f'], [0.35, '#f7f7f7'], [1, '#2ecc71']],
    zmin=0.45, zmax=1.0,
    colorbar=dict(title='AUC'),
))

fig_heat.update_layout(
    title=dict(
        text='<b>Сводный AUC Heatmap: Parity-k × подход × тестовое k</b>',
        x=0.5, font_size=14,
    ),
    height=600,
    xaxis=dict(side='top'),
    font=dict(size=11),
)
fig_heat.show()

In [ ]:
assert False

## 10. «Выпуклая» демонстрация роста AUC паренклитики

**Вопрос:** как увеличение порядка $k$ модели влияет на качество
паренклитики на датасетах с известным $k_\text{real}$?

### Сценарии

| # | Структура датасета | $d$ | $k_\text{real}$ | Ожидаемый паттерн |
|---|---|---|---|---|
| 1 | 1 блок k=7 + 3 шумовых признака | 10 | {7} | плоско до k=6, скачок на k=7 |
| 2 | блок k=5 + блок k=7 | 12 | {5, 7} | скачок на k=5, ещё на k=7 |
| 3 | три блока k=3, 5, 7 | 15 | {3, 5, 7} | лестница: k=3, 5, 7 |
| 4 | четыре блока k=1, 3, 5, 7 | 16 | {1, 3, 5, 7} | 4 ступеньки |

**Метод**: Parenclitic Features (fit only on class 0) + LogReg (основной);
Contrastive LLR (для сравнения).

> **Гипотеза**: AUC ≈ 0.5 при $k_\text{test} < k_\text{real}$
> и скачкообразно растёт при $k_\text{test} \geq$ очередной компоненты $k_\text{real}$.


In [17]:
def generate_parity_k_noisy(n, k, d_noise=0, seed=42):
    """1 блок Parity-k + d_noise шумовых признаков."""
    rng = np.random.default_rng(seed)
    X_sig = rng.uniform(-1, 1, size=(n, k))
    Y = np.sign(np.prod(X_sig, axis=1))
    mask = Y != 0
    X_sig = X_sig[mask]
    y_out = ((Y[mask] + 1) / 2).astype(int)
    if d_noise > 0:
        X_noise = rng.uniform(-1, 1, size=(X_sig.shape[0], d_noise))
        return np.hstack([X_sig, X_noise]), y_out
    return X_sig, y_out


DEMO_SCENARIOS = [
    {'name': 'S1: d=10, k_real={7}',     'k_reals': [7],       'n': 2000,
     'k_sweep': list(range(1, 8)), 'color': '#2980b9'},
    {'name': 'S2: d=12, k_real={5,7}',   'k_reals': [5, 7],    'n': 2000,
     'k_sweep': list(range(1, 8)), 'color': '#e74c3c'},
    {'name': 'S3: d=15, k_real={3,5,7}', 'k_reals': [3, 5, 7], 'n': 2000,
     'k_sweep': list(range(1, 8)), 'color': '#27ae60'},
    {'name': 'S4: d=16, k_real={1,3,5,7}', 'k_reals': [1, 3, 5, 7], 'n': 2000,
     'k_sweep': list(range(1, 8)), 'color': '#8e44ad'},
]


def make_dataset(sc, seed=42):
    k_reals = sc['k_reals']
    if len(k_reals) == 1:
        return generate_parity_k_noisy(sc['n'], k=k_reals[0], d_noise=3, seed=seed)
    components = [(k, k) for k in k_reals]
    return generate_mixed_parity(components, n=sc['n'], seed=seed)


for sc in DEMO_SCENARIOS:
    X_, y_ = make_dataset(sc)
    print(f"{sc['name']:30s}  d={X_.shape[1]}  n={len(y_):5}  balance={np.mean(y_):.3f}")


S1: d=10, k_real={7}            d=10  n= 2000  balance=0.494
S2: d=12, k_real={5,7}          d=12  n= 1042  balance=0.494
S3: d=15, k_real={3,5,7}        d=15  n= 2000  balance=0.497
S4: d=16, k_real={1,3,5,7}      d=16  n= 1236  balance=0.503


In [18]:
import time

def run_demo_sweep(sc, n_repeats=3, include_llr=True):
    X, y = make_dataset(sc)
    d = X.shape[1]
    k_sweep = sc['k_sweep']
    raw_feat = {k: [] for k in k_sweep}
    raw_llr  = {k: [] for k in k_sweep}

    for seed in range(n_repeats):
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=0.3, random_state=42 + seed, stratify=y
        )
        for k_test in k_sweep:
            # Parenclitic Features + LogReg
            try:
                m0 = MyModel(d=d, k=k_test)
                m0.fit_parallel(X_tr[y_tr == 0], n_jobs=-1)
                F_tr, _ = m0.get_feature_matrix_full_aggregated(X_tr)
                F_te, _ = m0.get_feature_matrix_full_aggregated(X_te)
                clf = Pipeline([('sc', StandardScaler()),
                                ('lr', LogisticRegression(max_iter=2000))])
                clf.fit(F_tr, y_tr)
                raw_feat[k_test].append(roc_auc_score(y_te, clf.predict_proba(F_te)[:, 1]))
            except Exception as e:
                raw_feat[k_test].append(0.5)
                print(f'  [Feat] {sc["name"]} k={k_test}: {e}')
            # Contrastive LLR
            if include_llr:
                try:
                    m0l = MyModel(d=d, k=k_test)
                    m1l = MyModel(d=d, k=k_test)
                    m0l.fit_parallel(X_tr[y_tr == 0], n_jobs=-1)
                    m1l.fit_parallel(X_tr[y_tr == 1], n_jobs=-1)
                    score = m1l.log_prob(X_te) - m0l.log_prob(X_te)
                    raw_llr[k_test].append(roc_auc_score(y_te, score))
                except Exception as e:
                    raw_llr[k_test].append(0.5)
                    print(f'  [LLR]  {sc["name"]} k={k_test}: {e}')

    def summ(r):
        return {k: {'auc_mean': np.mean(v), 'auc_std': np.std(v)} for k, v in r.items()}
    return {'feat': summ(raw_feat), 'llr': summ(raw_llr) if include_llr else {}}


print('run_demo_sweep ready.')


run_demo_sweep ready.


In [ ]:
demo_results = {}

for sc in DEMO_SCENARIOS:
    print(f'\n── {sc["name"]} ──')
    t0 = time.time()
    demo_results[sc['name']] = run_demo_sweep(sc, n_repeats=3, include_llr=True)
    elapsed = time.time() - t0
    res = demo_results[sc['name']]
    print(f'  Время: {elapsed:.1f}s')
    for ap, label in [('feat', 'Feat+LR'), ('llr', 'LLR    ')]:
        if res[ap]:
            row = '  '.join(f'k={k}: {res[ap][k]["auc_mean"]:.3f}' for k in sc['k_sweep'])
            print(f'  [{label}] {row}')

print('\n✓ Готово!')


── S1: d=10, k_real={7} ──
  Время: 31.4s
  [Feat+LR] k=1: 0.486  k=2: 0.487  k=3: 0.486  k=4: 0.486  k=5: 0.486  k=6: 0.486  k=7: 0.487
  [LLR    ] k=1: 0.464  k=2: 0.459  k=3: 0.459  k=4: 0.458  k=5: 0.459  k=6: 0.459  k=7: 0.459

── S2: d=12, k_real={5,7} ──
  Время: 91.0s
  [Feat+LR] k=1: 0.521  k=2: 0.523  k=3: 0.520  k=4: 0.517  k=5: 0.515  k=6: 0.512  k=7: 0.509
  [LLR    ] k=1: 0.525  k=2: 0.519  k=3: 0.515  k=4: 0.512  k=5: 0.510  k=6: 0.509  k=7: 0.508

── S3: d=15, k_real={3,5,7} ──


In [1]:
for sc in DEMO_SCENARIOS:
    k_sweep = sc['k_sweep']
    k_reals = sc['k_reals']
    res = demo_results[sc['name']]
    color = sc['color']
    fig = go.Figure()

    for ap, label, dash, sym in [
        ('feat', 'Parenclitic Features + LogReg', 'solid', 'circle'),
        ('llr',  'Contrastive Parenclitic (LLR)', 'dash',  'diamond'),
    ]:
        if not res[ap]: continue
        means = [res[ap][k]['auc_mean'] for k in k_sweep]
        stds  = [res[ap][k]['auc_std']  for k in k_sweep]
        fig.add_trace(go.Scatter(
            x=k_sweep, y=means,
            error_y=dict(type='data', array=stds, visible=True),
            mode='lines+markers', name=label,
            line=dict(color=hex_to_rgba(color, 0.8) if dash == 'dash' else color,
                      width=3, dash=dash),
            marker=dict(size=12, symbol=sym),
        ))

    fig.add_hline(y=0.5, line_dash='dot', line_color='gray', line_width=1.5)

    for kr in k_reals:
        fig.add_vline(x=kr, line_dash='dash', line_color='rgba(231,76,60,0.6)',
                      line_width=2,
                      annotation_text=f'k_real={kr}',
                      annotation_position='top right',
                      annotation_font=dict(size=11, color='rgba(200,50,50,0.9)'))

    # Аннотации регионов
    breakpoints = [0] + sorted(k_reals)
    for i, (lo, hi) in enumerate(zip(breakpoints, breakpoints[1:])):
        mid = (lo + hi) / 2 + 0.5
        lbl = 'слепой' if i == 0 else f'видит k={breakpoints[i]}'
        fig.add_annotation(x=mid, y=0.46, text=f'<i>{lbl}</i>',
                           showarrow=False, font=dict(size=10, color='gray'),
                           bgcolor='rgba(255,255,255,0.7)')

    fig.update_layout(
        title=dict(text=f'<b>{sc["name"]}</b><br>'
                        '<sup>Лестничный рост AUC при достижении каждого k_real</sup>',
                   x=0.5, font_size=14),
        xaxis=dict(title='Тестовое k (порядок модели)',
                   tickvals=k_sweep, ticktext=[str(k) for k in k_sweep]),
        yaxis=dict(title='AUC (ROC)', range=[0.35, 1.05]),
        plot_bgcolor='white', height=430,
        legend=dict(x=0.01, y=0.99), font=dict(size=12),
    )
    fig.update_xaxes(showgrid=True, gridcolor='rgba(200,200,200,0.3)')
    fig.update_yaxes(showgrid=True, gridcolor='rgba(200,200,200,0.4)')
    fig.show()


NameError: name 'DEMO_SCENARIOS' is not defined

In [ ]:
fig_bars = make_subplots(rows=2, cols=2,
    subplot_titles=[sc['name'] for sc in DEMO_SCENARIOS],
    vertical_spacing=0.18, horizontal_spacing=0.08)

for sc_idx, sc in enumerate(DEMO_SCENARIOS):
    row = sc_idx // 2 + 1
    col = sc_idx % 2 + 1
    k_sweep = sc['k_sweep']
    k_reals = sorted(sc['k_reals'])
    res = demo_results[sc['name']]['feat']
    color = sc['color']

    means = [res[k]['auc_mean'] for k in k_sweep]
    stds  = [res[k]['auc_std']  for k in k_sweep]

    def bar_color(k):
        n_reached = sum(1 for kr in k_reals if k >= kr)
        if n_reached == 0: return 'lightgray'
        if n_reached < len(k_reals): return hex_to_rgba(color, 0.55)
        return color

    def bar_label(k, v):
        n_reached = sum(1 for kr in k_reals if k >= kr)
        note = {0: 'blind', len(k_reals): 'full'}.get(n_reached, 'partial')
        return f'{v:.3f}'

    fig_bars.add_trace(
        go.Bar(
            x=[str(k) for k in k_sweep], y=means,
            error_y=dict(type='data', array=stds, visible=True),
            marker_color=[bar_color(k) for k in k_sweep],
            marker_line_color='white', marker_line_width=1.5,
            text=[bar_label(k, v) for k, v in zip(k_sweep, means)],
            textposition='outside', showlegend=False,
        ), row=row, col=col)

    fig_bars.add_hline(y=0.5, line_dash='dot', line_color='gray',
                       line_width=1.5, row=row, col=col)
    for kr in k_reals:
        idx = k_sweep.index(kr)
        fig_bars.add_vline(x=idx - 0.5, line_dash='dash',
                           line_color='rgba(231,76,60,0.7)', line_width=1.5,
                           row=row, col=col)

    fig_bars.update_xaxes(title_text='Тестовое k', row=row, col=col)
    fig_bars.update_yaxes(title_text='AUC', range=[0.3, 1.22],
                          showgrid=True, gridcolor='rgba(200,200,200,0.4)',
                          row=row, col=col)

fig_bars.update_layout(
    title=dict(text='<b>Паренклитика + LogReg: лестничный рост AUC</b><br>'
                    '<sup>Серый = слепой | Полупрозрачный = видит часть компонент | Тёмный = видит всё</sup>',
               x=0.5, font_size=14),
    plot_bgcolor='white', height=750, bargap=0.2, font=dict(size=11),
)
fig_bars.update_xaxes(showgrid=False)
fig_bars.show()


In [ ]:
fig_ov = go.Figure()

for sc in DEMO_SCENARIOS:
    k_sweep = sc['k_sweep']
    res = demo_results[sc['name']]['feat']
    means = [res[k]['auc_mean'] for k in k_sweep]
    stds  = [res[k]['auc_std']  for k in k_sweep]
    fig_ov.add_trace(go.Scatter(
        x=k_sweep, y=means,
        error_y=dict(type='data', array=stds, visible=True),
        mode='lines+markers', name=sc['name'],
        line=dict(color=sc['color'], width=2.5), marker=dict(size=9),
    ))

fig_ov.add_hline(y=0.5, line_dash='dot', line_color='gray', line_width=1.5)

for kr in sorted({k for sc in DEMO_SCENARIOS for k in sc['k_reals']}):
    fig_ov.add_vline(x=kr, line_dash='dash',
                     line_color='rgba(100,100,100,0.35)', line_width=1.5,
                     annotation_text=f'k={kr}',
                     annotation_position='top',
                     annotation_font=dict(size=10, color='gray'))

fig_ov.update_layout(
    title=dict(text='<b>Все сценарии: лестничный рост AUC (Parenclitic Feat + LogReg)</b><br>'
                    '<sup>Вертикальные линии = позиции k_real компонент</sup>',
               x=0.5, font_size=14),
    xaxis=dict(title='Тестовое k', tickvals=list(range(1, 8))),
    yaxis=dict(title='AUC (ROC)', range=[0.35, 1.05]),
    plot_bgcolor='white', height=480,
    legend=dict(x=0.01, y=0.99), font=dict(size=12),
)
fig_ov.update_xaxes(showgrid=True, gridcolor='rgba(200,200,200,0.3)')
fig_ov.update_yaxes(showgrid=True, gridcolor='rgba(200,200,200,0.4)')
fig_ov.show()


In [ ]:
from IPython.display import Markdown, display

rows_md = []
k_all = list(range(1, 8))
ks_hdr = ' | '.join(f'k={k}' for k in k_all)
rows_md.append(f'| Сценарий | Подход | {ks_hdr} |')
rows_md.append('|---|---|' + '|'.join(['---'] * len(k_all)) + '|')

for sc in DEMO_SCENARIOS:
    k_reals = sorted(sc['k_reals'])
    for ap_key, ap_label in [('feat', '**Feat+LogReg**'), ('llr', 'LLR')]:
        res = demo_results[sc['name']][ap_key]
        def fmt(k):
            v = res[k]['auc_mean']
            n_r = sum(1 for kr in k_reals if k >= kr)
            if n_r == 0:             return f'{v:.3f}'
            if n_r < len(k_reals):   return f'_{v:.3f}_'
            return f'**{v:.3f}**'
        cells = ' | '.join(fmt(k) for k in k_all)
        rows_md.append(f'| {sc["name"]} | {ap_label} | {cells} |')

rows_md.append('')
rows_md.append('> обычный=слепой | *курсив*=частично | **жирный**=видит всё')
display(Markdown('\n'.join(rows_md)))


In [ ]:
heat_rows = []
for sc in DEMO_SCENARIOS:
    for ap_key, ap_label in [('feat', 'Feat+LR'), ('llr', 'LLR')]:
        res = demo_results[sc['name']][ap_key]
        row_z = [res.get(k, {}).get('auc_mean', None) for k in range(1, 8)]
        row_t = [f"{v:.3f}" if v is not None else '' for v in row_z]
        heat_rows.append({'label': f"{sc['name']} / {ap_label}",
                          'z': row_z, 't': row_t, 'k_reals': sc['k_reals']})

fig_heat = go.Figure(go.Heatmap(
    z=[r['z'] for r in heat_rows],
    x=[f'k={k}' for k in range(1, 8)],
    y=[r['label'] for r in heat_rows],
    text=[r['t'] for r in heat_rows],
    texttemplate='%{text}', textfont=dict(size=11),
    colorscale=[[0,'#d9534f'],[0.3,'#f5a623'],[0.5,'#f7f7f7'],[1.0,'#27ae60']],
    zmin=0.45, zmax=1.0,
    colorbar=dict(title='AUC', tickformat='.2f'),
))

for row_idx, r in enumerate(heat_rows):
    for kr in r['k_reals']:
        col_idx = kr - 1
        fig_heat.add_shape(
            type='rect',
            x0=col_idx - 0.5, x1=col_idx + 0.5,
            y0=row_idx - 0.5, y1=row_idx + 0.5,
            line=dict(color='black', width=2),
            fillcolor='rgba(0,0,0,0)',
        )

fig_heat.update_layout(
    title=dict(
        text='<b>Heatmap AUC(тест. k) — демо-сценарии</b><br>'
             '<sup>Рамка = позиция компоненты k_real | Красный=слепой | Зелёный=информативный</sup>',
        x=0.5, font_size=14),
    xaxis=dict(side='top'), height=560, font=dict(size=11),
)
fig_heat.show()


### Выводы раздела 10

#### Что подтверждает эксперимент

**1. «Лестничный» прирост AUC:**
Каждый раз, когда тестовое $k$ достигает очередной компоненты $k_\text{real}$,
AUC делает скачок. Это подтверждает:
- паренклитика с порядком $k' < k_\text{real}$ **слепа** — все признаки
  из нормальной модели одинаково информативны для обоих классов;
- при $k' = k_\text{real}$ условные плотности начинают различаться → сигнал детектируется.

**2. Mixed Parity → $m$ ступенек:**
Для $m$ компонент с порядками $k_1 < k_2 < \ldots < k_m$ AUC-кривая имеет
ровно $m$ скачков. Сценарий 4 с $k_\text{real} = \{1,3,5,7\}$ демонстрирует
**четыре** отчётливые ступеньки.

**3. Parenclitic Features vs Contrastive LLR:**

| | Feat + LogReg | Contrastive LLR |
|---|---|---|
| **Порог** | прыжок на $k_\text{real}$ | то же |
| **Высота прыжка** | умеренная (LogReg линеен) | сопоставимая |
| **Плюс** | использует богатые признаки | нативный, без внешнего clf |

**4. Практическая рекомендация:**
> Если порядок взаимодействия в данных неизвестен — используй паренклитику
> как **детектор $k_\text{real}$**: обучи с $k = 1, 2, \ldots, k_\text{max}$
> и найди точку скачка AUC. Это даёт оценку истинного $k_\text{real}$ в данных.

**5. Ограничение гауссовой параметрики:**
На мультипликативных структурах Parity-$k$ высота скачков ограничена, потому что
линейная регрессия плохо аппроксимирует $\mathbb{E}[x_k \mid x_1,\ldots,x_{k-1}]$.
На **гауссовых** данных с $k$-порядковыми линейными корреляциями скачки
будут значительно более выраженными.


## 9. Выводы

### 9.1 Паренклитика: два подхода

**Contrastive Parenclitic (LLR)** — нативный подход:
- Обучает $M_0$ и $M_1$ на каждом классе отдельно
- Скор $= \log p_1(x) - \log p_0(x)$; полностью генеративный, без внешнего классификатора

**Parenclitic Features (fit class 0)** — аномальный подход:
- $M_0$ обучается **только** на классе 0 («норма»)
- Признаки = log-условные плотности нормальной модели; смысл: «насколько $x$ отклоняется от нормы»
- Требует внешнего классификатора

### 9.2 Mixed Parity: поведение подходов

Mixed Parity содержит **сигналы двух порядков** одновременно ($k_1$ и $k_2 > k_1$).
Ожидаемое поведение при sweep тестового $k'$:

| Диапазон тест. $k'$ | Ожидание |
|---|---|
| $k' < k_1$ | AUC ≈ 0.5 — слеп к обоим сигналам |
| $k_1 \le k' < k_2$ | AUC > 0.5 — улавливает компоненту $k_1$ |
| $k' \ge k_2$ | AUC ещё выше — видит обе компоненты |

### 9.3 Влияние ML-модели

| Подход | LogReg | RBF-SVM | GBM | Комментарий |
|---|---|---|---|---|
| **Contrastive LLR** | — | — | — | Нативный, не зависит от ML |
| **Synolitic Native** | — | — | — | Нативный, Мёбиус |
| **Parenclitic Feat.** | Умеренный | Выше | Выше | Нелинейный clf лучше |
| **Synolitic Feat.** | Умеренный | **Высокий** | Высокий | RBF видит мультипликативную структуру |

**Ключевое наблюдение**: для Parity/Mixed структур с мультипликативными взаимодействиями
наиболее мощна комбинация **Synolitic Features + RBF-SVM**:
- Синолитика учит $p(y | x_T)$ дискриминативно — прямо отражает метку;
- RBF-SVM может выучить нелинейную границу $\operatorname{sign}(x_1 \cdot x_2 \cdots x_k)$.

### 9.4 Гауссовое ограничение паренклитики

Условное распределение $p(x_i \mid x_I)$ в Parity-$k$ нелинейно.
Линейная регрессия MyModel плохо аппроксимирует $E[x_k \mid x_1,\ldots,x_{k-1}] = f(x_1 \cdots x_{k-1})$.

Паренклитика выигрывает там, где взаимодействия **линейны** (гауссовы данные,
линейные коэффициенты взаимодействия). На Parity-$k$ она даёт частичный сигнал
через LLR, но уступает нелинейной синолитике.

### 9.5 Итог

> - **LLR** и **Synolitic Native** — нативные нулевые базисы (без дополнительного обучения).
> - При добавлении **нелинейного классификатора** (RBF-SVM, GBM) оба Features-подхода значительно улучшаются.
> - **Synolitic Features + RBF-SVM** — сильнейшая конфигурация на Parity-структурах.
> - **Parenclitic (LLR)** — конкурентоспособен на линейных задачах, не требует разметки классов в отдельных запусках.
